<a href="https://colab.research.google.com/github/shravan1808/DeepLearningProjects/blob/main/Capstone/Capstone_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader,random_split

In [ ]:
torch.manual_seed(42)
X = torch.randn(120,256)
y = torch.randint(0,3,size=(120,))

In [ ]:
class SyntheticDataset(Dataset):
  def __init__(self,X,y):
    self.X=X
    self.y=y

  def __len__(self):
    return len(self.X)

  def __getitem__(self,idx):
    return self.X[idx],self.y[idx]

dataset = SyntheticDataset(X,y)

train_data,val_data = random_split(dataset,[96,24])

train_dataloader = DataLoader(train_data,batch_size=16,shuffle=True)
val_dataloader = DataLoader(val_data,batch_size=16,shuffle=False)

In [ ]:
class RegularizedClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(256,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(64,3)
    )

  def forward(self,X):
    X = self.network(X)
    return X

In [ ]:
model = RegularizedClassifier()

In [ ]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.005,
    weight_decay=1e-4
)
loss_fn = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=20,eta_min=1e-5)

In [ ]:
loss_history = []
best_loss = float('inf')
for epoch in range(20):
  epoch_loss = 0
  model.train()
  for X_batch,y_batch in train_dataloader:
    optimizer.zero_grad()
    output = model(X_batch)
    loss = loss_fn(output,y_batch)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
    optimizer.step()
    epoch_loss += loss.item()
  avg_loss = epoch_loss/len(train_dataloader)
  loss_history.append(avg_loss)
  model.eval()
  with torch.no_grad():
    val_loss = 0
    correct = 0
    total = 0
    for X,y in val_dataloader:
      val_output = model(X)
      predicted = torch.argmax(val_output,dim=1)
      batch_val_loss = loss_fn(val_output, y)
      val_loss += batch_val_loss.item()
      total += y.size(0)
      correct += (predicted==y).sum().item()
    accuracy = correct/total
  avg_val_loss = val_loss/len(val_dataloader)
  if avg_val_loss < best_loss:
    best_loss=avg_val_loss
    checkpoint = {
        'model_state_dict':model.state_dict(),
        'epoch':epoch+1,
        'best_loss':best_loss
    }
    torch.save(checkpoint,'best_model.pt')
  print(
    f"Epoch [{epoch+1}/20] "
    f"Train Loss: {avg_loss:.4f} | "
    f"Val Loss: {avg_val_loss:.4f} | "
    f"Val Accuracy: {accuracy*100:.2f}% | "
    f"LR: {optimizer.param_groups[0]['lr']:.6f}"
  )
  scheduler.step()


Epoch [1/20] Train Loss: 1.1442 | Val Loss: 1.0895 | Val Accuracy: 37.50% | LR: 0.005000
Epoch [2/20] Train Loss: 0.9499 | Val Loss: 1.0854 | Val Accuracy: 45.83% | LR: 0.004969
Epoch [3/20] Train Loss: 0.7598 | Val Loss: 1.1038 | Val Accuracy: 45.83% | LR: 0.004878
Epoch [4/20] Train Loss: 0.5366 | Val Loss: 1.1419 | Val Accuracy: 45.83% | LR: 0.004728
Epoch [5/20] Train Loss: 0.3938 | Val Loss: 1.1830 | Val Accuracy: 50.00% | LR: 0.004523
Epoch [6/20] Train Loss: 0.2915 | Val Loss: 1.2429 | Val Accuracy: 50.00% | LR: 0.004269
Epoch [7/20] Train Loss: 0.1488 | Val Loss: 1.3023 | Val Accuracy: 50.00% | LR: 0.003972
Epoch [8/20] Train Loss: 0.1346 | Val Loss: 1.3591 | Val Accuracy: 50.00% | LR: 0.003638
Epoch [9/20] Train Loss: 0.1113 | Val Loss: 1.3749 | Val Accuracy: 50.00% | LR: 0.003276
Epoch [10/20] Train Loss: 0.0729 | Val Loss: 1.4138 | Val Accuracy: 50.00% | LR: 0.002895
Epoch [11/20] Train Loss: 0.0767 | Val Loss: 1.4204 | Val Accuracy: 45.83% | LR: 0.002505
Epoch [12/20] Train

In [ ]:
print("Best Validation Loss:", best_loss)
print("Checkpoint Saved: best_model.pt")

Best Validation Loss: 1.0854058861732483
Checkpoint Saved: best_model.pt


In [ ]:
def architecture_preview():

    dataset_1 = {
        "type": "SyntheticDataset",
        "samples": 120,
        "features": 256,
        "classes": 3,
        "batch_size": 16
    }

    model_2 = {
    "type": "RegularizedClassifier",
    "architecture": "256 -> 128 -> 64 -> 3",
    "batch_norm": True,
    "dropout": 0.4,
    "activation": "ReLU"
    }

    optimizer_3 = {
    "type": "Adam",
    "learning_rate": 0.005,
    "weight_decay": 1e-4
    }

    scheduler_4 = {
    "type": "CosineAnnealingLR",
    "T_max": 20,
    "eta_min": 1e-5
    }

    trainer_5 = {
    "epochs": 20,
    "batch_size": 16,
    "loss_function": "CrossEntropyLoss",
    "optimizer": optimizer_3,
    "scheduler": scheduler_4,
    "gradient_clipping": 1.0
    }

    checkpoint_6 = {
    "type": "Best Model Checkpoint",
    "file": "best_model.pt",
    "contents": [
        "model_state_dict",
        "epoch",
        "best_loss"
    ]
    }

    return {
        "dataset_1": dataset_1,
        "model_2": model_2,
        "optimizer_3": optimizer_3,
        "scheduler_4": scheduler_4,
        "trainer_5": trainer_5,
        "checkpoint_6": checkpoint_6
    }